# LumaCam: Exposure-Conditioned Attention U-Net GAN Training
### Final Year Capstone Project — Team 21 (SCTCE)
> **Authors**: Sreenandan S (263), Sreelakshmi A (262), Sreeveni R (264)  
> **Guide**: Asst. Prof. Arya Sreedhar  
> **Objective**: Train the real-time Low-Light Image Enhancement (LLIE) Exposure-Conditioned Attention U-Net GAN on the See-in-the-Dark (SID) Sony dataset using Kaggle's free Tesla P100 GPU.

---
### Key Architecture Features:
1. **Packed 4-Channel Bayer Pipeline**: Directly processes raw sensor data $(H/2, W/2, 4)$ before demosaicing to avoid noise amplification.
2. **Dynamic FiLM Latent Conditioning**: User exposure scalar ($\alpha$) modulates the U-Net bottleneck features in real time.
3. **Attention Gates (AG)**: Spatial attention gates on skip connections suppress dark sensor shot noise.
4. **PatchGAN Discriminator**: $70 \times 70$ Markovian patch evaluation prevents CNN over-smoothing and recovers realistic micro-textures.
5. **Composite Objective**: Joint optimization with Pixel $L_1$ + Wang et al. 5-Scale MS-SSIM + Conditional Adversarial Loss.

In [ ]:
# =============================================================================
# 1. HARDWARE DETECTION & DEPENDENCY SETUP
# =============================================================================
import os
import sys
import time
from pathlib import Path
import torch

print(f"[System] PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[System] GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[System WARNING] No GPU detected. In Kaggle, click Settings -> Accelerator -> GPU P100/T4.")

# Install rawpy for .ARW sensor file parsing and quality metrics
!pip install -q rawpy torchmetrics
print("[System] Dependencies successfully installed!")

In [ ]:
# =============================================================================
# 2. CLONE OR MOUNT LUMACAM REPOSITORY
# =============================================================================
# Clone the latest development branch containing our architecture
if not Path("LumaCam").exists():
    !git clone -b dev https://github.com/Sreenandan2004/LumaCam.git
else:
    !cd LumaCam && git checkout dev && git pull origin dev

# Add repository root to python search path
repo_path = Path("LumaCam").resolve()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from model.networks.generator import ExposureConditionedAttentionUNet
from model.networks.discriminator import PatchGANDiscriminator
from model.losses.composite_loss import CompositeLoss, AdversarialLoss, MSSSIMLoss
from model.datasets.sid_dataset import SIDSonyDataset, DummySIDDataset

print("[System] Successfully imported LumaCam neural modules from dev branch!")

In [ ]:
# =============================================================================
# 3. LOCATE SEE-IN-THE-DARK (SID) SONY DATASET
# =============================================================================
# Kaggle datasets are mounted under /kaggle/input/
dataset_root = None
possible_paths = [
    Path("/kaggle/input/see-in-the-dark-dataset/Sony"),
    Path("/kaggle/input/see-in-the-dark-sony/Sony"),
    Path("/kaggle/input/sid-sony/Sony"),
    Path("/kaggle/input/seeinthedark/Sony"),
]

# Automatic scanner across all input directories
input_dir = Path("/kaggle/input")
if input_dir.exists():
    for path in input_dir.rglob("short"):
        if path.is_dir():
            dataset_root = path.parent
            print(f"[Dataset] Found SID Sony directory at: {dataset_root}")
            break

PATCH_SIZE = 512
BATCH_SIZE = 4

if dataset_root and (dataset_root / "short").exists():
    print(f"[Dataset] Initializing SIDSonyDataset using real .ARW files...")
    train_dataset = SIDSonyDataset(
        data_root=dataset_root,
        patch_size=PATCH_SIZE,
        is_train=True,
    )
    val_dataset = SIDSonyDataset(
        data_root=dataset_root,
        patch_size=PATCH_SIZE,
        is_train=False,
        max_samples=20,
    )
else:
    print("[Dataset NOTE] No SID Sony dataset attached in /kaggle/input.")
    print("[Dataset NOTE] Using DummySIDDataset with Poisson-Gaussian sensor noise for demonstration.")
    print("[Dataset NOTE] (To attach real dataset: Click '+ Add Input' -> Search 'See in the Dark Sony')")
    train_dataset = DummySIDDataset(num_samples=64, patch_size=PATCH_SIZE, seed=42)
    val_dataset = DummySIDDataset(num_samples=8, patch_size=PATCH_SIZE, seed=1337)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2 if os.name != 'nt' else 0,
    pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

print(f"[Dataset] Training samples: {len(train_dataset)} | Batches per epoch: {len(train_loader)}")
print(f"[Dataset] Validation samples: {len(val_dataset)}")

In [ ]:
# =============================================================================
# 4. INITIALIZE MODELS, LOSSES & OPTIMIZERS
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generator: Exposure-Conditioned Attention U-Net
generator = ExposureConditionedAttentionUNet(
    in_channels=4,
    out_channels=3,
    base_channels=32,
    cond_dim=1,
    use_pixel_shuffle=True,
).to(device)

# Discriminator: 70x70 Markovian PatchGAN (4 Bayer upsampled + 3 RGB target = 7 channels)
discriminator = PatchGANDiscriminator(in_channels=7, base_channels=64).to(device)

# Composite Training Objective: L1 + MS-SSIM + GAN
composite_loss = CompositeLoss(
    lambda_l1=100.0,
    lambda_ms_ssim=10.0,
    lambda_adv=1.0,
).to(device)
adv_loss = AdversarialLoss().to(device)

# Optimizers (Adam with beta1=0.5 for GAN stability)
optimizer_g = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.5, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.999))

# Training Hyperparameters & Cosine Annealing Schedulers
EPOCHS = 20  # Set to 20-50 for high fidelity
WARMUP_EPOCHS = 5  # Pure L1 + MS-SSIM reconstruction warmup before turning on PatchGAN
scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_g, T_max=EPOCHS, eta_min=1e-6)
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_d, T_max=EPOCHS, eta_min=1e-6)

# Automatic Mixed Precision (AMP) on Tesla P100 with stable initial scale
use_amp = torch.cuda.is_available()
scaler_g = torch.amp.GradScaler('cuda', init_scale=1024.0, enabled=use_amp)
scaler_d = torch.amp.GradScaler('cuda', init_scale=1024.0, enabled=use_amp)

gen_params = sum(p.numel() for p in generator.parameters()) / 1e6
disc_params = sum(p.numel() for p in discriminator.parameters()) / 1e6
print(f"[Model] Generator Parameters: {gen_params:.2f}M")
print(f"[Model] Discriminator Parameters: {disc_params:.2f}M")
print(f"[Training Config] Warmup epochs: {WARMUP_EPOCHS} | Schedulers: CosineAnnealingLR (1e-4 -> 1e-6)")


In [ ]:
# =============================================================================
# 5. TRAINING & VALIDATION LOOP WITH BENCHMARK METRICS
# =============================================================================
def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = torch.mean((pred - target) ** 2).item()
    if mse <= 1e-10:
        return 100.0
    return float(20.0 * torch.log10(torch.tensor(1.0) / torch.sqrt(torch.tensor(mse))).item())

save_dir = Path("checkpoints")
save_dir.mkdir(exist_ok=True)

history = {"epoch": [], "g_loss": [], "d_loss": [], "val_psnr": [], "val_ssim": []}
best_psnr = -1.0
ms_ssim_eval = MSSSIMLoss().to(device)

print(f"[Training] Starting LumaCam GAN training for {EPOCHS} epochs...")
print(f"[Training] Warmup Phase: Epochs 1-{WARMUP_EPOCHS} use pure reconstruction (L1 + MS-SSIM); PatchGAN engages at Epoch {WARMUP_EPOCHS+1}!")

for epoch in range(1, EPOCHS + 1):
    t_start = time.time()
    generator.train()
    discriminator.train()
    in_warmup = (epoch <= WARMUP_EPOCHS)
    run_g, run_d = 0.0, 0.0

    for bayer_in, gt_rgb, alpha in train_loader:
        bayer_in = bayer_in.to(device)
        gt_rgb = gt_rgb.to(device)
        alpha = alpha.to(device)

        # 1. Update Discriminator (PatchGAN) — active only after warmup
        d_loss_val = 0.0
        if not in_warmup:
            optimizer_d.zero_grad()
            with torch.amp.autocast('cuda', enabled=use_amp):
                real_pred = discriminator(bayer_in, gt_rgb)
                with torch.no_grad():
                    fake_rgb, _ = generator(bayer_in, alpha)
                fake_pred = discriminator(bayer_in, fake_rgb.detach())
            d_loss = adv_loss.discriminator_loss(real_pred.float(), fake_pred.float())
            scaler_d.scale(d_loss).backward()
            scaler_d.unscale_(optimizer_d)
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=5.0)
            scaler_d.step(optimizer_d)
            scaler_d.update()
            d_loss_val = d_loss.item()

        # 2. Update Generator
        optimizer_g.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            gen_rgb, _ = generator(bayer_in, alpha)
            disc_fake = discriminator(bayer_in, gen_rgb) if not in_warmup else None

        # Calculate composite loss in float32 outside autocast to prevent float16 overflow
        g_losses = composite_loss(
            gen_rgb.float(),
            gt_rgb.float(),
            disc_fake.float() if disc_fake is not None else None,
            use_adv=(not in_warmup),
        )
        g_loss = g_losses['total']

        scaler_g.scale(g_loss).backward()
        scaler_g.unscale_(optimizer_g)
        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=5.0)
        scaler_g.step(optimizer_g)
        scaler_g.update()

        run_d += d_loss_val
        run_g += g_loss.item()

    scheduler_g.step()
    if not in_warmup:
        scheduler_d.step()

    avg_d = run_d / len(train_loader)
    avg_g = run_g / len(train_loader)
    elapsed = time.time() - t_start

    # Validation Evaluation
    generator.eval()
    val_psnrs, val_ssims = [], []
    with torch.no_grad():
        for v_bayer, v_gt, v_alpha in val_loader:
            v_bayer, v_gt, v_alpha = v_bayer.to(device), v_gt.to(device), v_alpha.to(device)
            v_pred, _ = generator(v_bayer, v_alpha)
            v_pred = torch.clamp(v_pred, 0.0, 1.0)
            val_psnrs.append(compute_psnr(v_pred, v_gt))
            try:
                ms_val = ms_ssim_eval(v_pred, v_gt).item()
                val_ssims.append(max(0.0, 1.0 - ms_val))
            except Exception:
                pass

    m_psnr = sum(val_psnrs) / max(len(val_psnrs), 1)
    m_ssim = sum(val_ssims) / max(len(val_ssims), 1) if val_ssims else 0.0

    history["epoch"].append(epoch)
    history["g_loss"].append(avg_g)
    history["d_loss"].append(avg_d)
    history["val_psnr"].append(m_psnr)
    history["val_ssim"].append(m_ssim)

    star = ""
    if m_psnr > best_psnr:
        best_psnr = m_psnr
        torch.save({
            "epoch": epoch,
            "generator_state_dict": generator.state_dict(),
            "val_psnr": m_psnr,
            "val_ssim": m_ssim,
        }, save_dir / "lumacam_best.pt")
        star = " [BEST]"

    phase_str = "[WARMUP]" if in_warmup else "[GAN]"
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] {phase_str} ({elapsed:.1f}s) | G_Loss: {avg_g:.3f} | D_Loss: {avg_d:.3f} | Val PSNR: {m_psnr:.2f} dB | Val SSIM: {m_ssim:.4f}{star}")

print(f"\n[Training Complete] Peak Validation PSNR: {best_psnr:.2f} dB")


In [ ]:
# =============================================================================
# 6. PLOT TRAINING CURVES (FOR PRESENTATION)
# =============================================================================
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss Curves with NaN safety
epochs_arr = np.array(history["epoch"])
g_loss_arr = np.array(history["g_loss"])
d_loss_arr = np.array(history["d_loss"])

valid_g = ~np.isnan(g_loss_arr)
if valid_g.any():
    ax1.plot(epochs_arr[valid_g], g_loss_arr[valid_g], 'b-', linewidth=2, label="Generator Total Loss")
valid_d = ~np.isnan(d_loss_arr) & (d_loss_arr > 0)
if valid_d.any():
    ax1.plot(epochs_arr[valid_d], d_loss_arr[valid_d], 'r--', linewidth=2, label="PatchGAN D Loss (Post-Warmup)")
ax1.set_title("LumaCam GAN Training Loss", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss Value")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. PSNR Convergence
ax2.plot(history["epoch"], history["val_psnr"], 'g-o', linewidth=2, label="Validation PSNR (dB)")
ax2.axhline(28.0, color='orange', linestyle=':', linewidth=2, label="Target Baseline (28 dB)")
ax2.set_title("Validation Peak Signal-to-Noise Ratio (PSNR)", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("PSNR (dB)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=300)
plt.show()
print("[Visuals] Saved training curves to training_curves.png!")


In [ ]:
# =============================================================================
# 7. VISUAL ENHANCEMENT COMPARISON (PITCH BLACK VS ENHANCED VS GT)
# =============================================================================
import numpy as np

# Fetch one validation sample
generator.eval()
sample_bayer, sample_gt, sample_alpha = next(iter(val_loader))
sample_bayer = sample_bayer.to(device)
sample_alpha = sample_alpha.to(device)

with torch.no_grad():
    enhanced, attn_maps = generator(sample_bayer, sample_alpha, return_attention_maps=True)
    enhanced = torch.clamp(enhanced, 0.0, 1.0)

# Format for display
# Input preview: average RGGB channels to show dark raw sensor view
input_preview = sample_bayer[0].mean(dim=0).cpu().numpy()
input_preview = np.clip(input_preview, 0.0, 1.0)

enhanced_rgb = enhanced[0].permute(1, 2, 0).cpu().numpy()
ground_truth_rgb = sample_gt[0].permute(1, 2, 0).cpu().numpy()

sample_psnr = compute_psnr(enhanced[0:1].cpu(), sample_gt[0:1])

# Plot 3-Panel Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(input_preview, cmap='gray')
axes[0].set_title(f"Input Bayer Raw (Short Exp)\nAlpha = {sample_alpha.item():.1f}x", fontsize=12)
axes[0].axis('off')

axes[1].imshow(enhanced_rgb)
axes[1].set_title(f"LumaCam Enhanced (Ours)\nPSNR: {sample_psnr:.2f} dB", fontsize=12, fontweight='bold', color='green')
axes[1].axis('off')

axes[2].imshow(ground_truth_rgb)
axes[2].set_title("Ground Truth Reference\n(Long Exposure tripod)", fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.savefig("enhancement_comparison.png", dpi=300)
plt.show()
print("[Visuals] Saved 3-panel comparison to enhancement_comparison.png!")

In [ ]:
# =============================================================================
# 8. ATTENTION GATE HEATMAP VISUALIZATION
# =============================================================================
if attn_maps is not None:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for idx, (a_map, ax) in enumerate(zip(attn_maps, axes), start=1):
        # Soft attention weights in [0, 1]
        att_np = a_map[0, 0].cpu().numpy()
        im = ax.imshow(att_np, cmap='inferno')
        ax.set_title(f"Attention Gate {idx} Heatmap\n(Suppressing Sensor Noise)", fontsize=11)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig("attention_maps.png", dpi=300)
    plt.show()
    print("[Visuals] Saved Attention Gate heatmaps to attention_maps.png!")

In [ ]:
# =============================================================================
# 9. DOWNLOAD CHECKPOINT FOR ONNX / TFLITE CONVERSION
# =============================================================================
from IPython.display import FileLink

best_ckpt = Path("checkpoints/lumacam_best.pt")
if best_ckpt.exists():
    print(f"[Download] Click the link below to download your trained checkpoint:")
    display(FileLink(str(best_ckpt)))
else:
    print("[Download] Checkpoint file not found yet.")